# Historical archive: gemma2-2b-fewshot

**Status: not independently reproduced.**

Checkpoint / purpose: `google/gemma-2-2b`.

Historical code, not independently reproduced. Saved settings can differ from report labels; notebook filenames are not authoritative configurations. Exploratory classification and prompting variants. The report describes NaN failures; no verified comparable performance claim is made.

Publication edits only: credentials use `HF_TOKEN`; personal absolute paths were relocated; saved outputs, attachments and runtime metadata were removed. Historical installation cells and local data dependencies are retained. Review code before running; it can download models, install packages and start expensive training. See `docs/experiment_catalogue.md` and the new pipeline for supported experiments.


In [ ]:
#!pip install datasets
#!pip install bitsandbytes
#!pip install transformers
#!pip install peft

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
import torch
import torch.nn as nn
from sklearn.preprocessing import MultiLabelBinarizer
from datasets import Dataset
import pandas as pd
import os
import numpy as np
from peft import get_peft_model, LoraConfig, TaskType

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
# Multilabel binarizer to transform article labels into binary vectors
article_mapping = {
    'Article 1: Obligation to respect human rights': 0,
    'Article 2: Right to life': 1,
    'Article 3: Prohibition of torture': 2,
    'Article 4: Prohibition of slavery and forced labour': 3,
    'Article 5: Right to liberty and security': 4,
    'Article 6: Right to a fair trial': 5,
    'Article 7: No punishment without law': 6,
    'Article 8: Right to respect for private and family life': 7,
    'Article 9: Freedom of thought, conscience and religion': 8,
    'Article 10: Right to freedom of speech and expression': 9,
    'No Articles Violated': 10
}

num_labels = len(article_mapping)  # number of unique labels
print(num_labels)

In [ ]:
model_name = "google/gemma-2-2b"
access_token = __import__('os').environ['HF_TOKEN']  # Replace with your actual token

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b", token=access_token)

# Define a custom model class with a sigmoid output layer
class MultilabelClassificationModel(nn.Module):
    def __init__(self, model, num_labels):
        super(MultilabelClassificationModel, self).__init__()
        self.model = model
        self.num_labels = num_labels
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = self.sigmoid(logits)

        loss = None
        if labels is not None:
            #print(f"logits shape: {logits.shape}")
            #print(f"labels shape: {labels.shape}")
            loss_fct = nn.BCEWithLogitsLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1, self.num_labels).float())

        return {'loss': loss, 'logits': logits, 'probs': probs}


In [ ]:
# Load the pretrained model
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels, token=access_token)

In [ ]:
# Configure LoRA
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",], 
    r=8,  # Low-rank dimension
    lora_alpha=32, 
    lora_dropout=0.1,
    bias="none"
)

In [ ]:
# Apply LoRA to the model
model_with_lora = get_peft_model(base_model, peft_config)
model_with_lora.gradient_checkpointing_enable()

In [ ]:
# Initialize the custom model
model = MultilabelClassificationModel(model_with_lora, num_labels)

model_with_lora.print_trainable_parameters()  # Print number of trainable parameters

In [ ]:
# Load data
train = pd.read_parquet('ecthr_a_article_desc_train.parquet')
test = pd.read_parquet('ecthr_a_article_desc_test.parquet')
validation = pd.read_parquet('ecthr_a_article_desc_validation.parquet')

In [ ]:
# Convert DataFrame to Dataset
train_dataset = Dataset.from_pandas(train)
test_dataset = Dataset.from_pandas(test)
validation_dataset = Dataset.from_pandas(validation)

# Define preprocessing function
def preprocess_function(examples):
    instructions = "Based on the following case, tell me which articles of the ECHR may have been violated:"
    inputs = examples["text"]
    outputs = examples["articles"]

    inputs = [f"{instructions} {text}" for text in inputs]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    mlb = MultiLabelBinarizer(classes=list(article_mapping.keys()))
    labels = mlb.fit_transform(outputs)
    model_inputs["labels"] = labels.tolist()

    return model_inputs


In [ ]:
# Random 200 samples to perform few-shot learning
few_shot_samples = train.sample(n=2000, random_state=42)
few_shot_dataset = Dataset.from_pandas(few_shot_samples)

tokenized_dataset = few_shot_dataset.map(preprocess_function, batched=True)

In [ ]:
# Evaluating on first 10 examples of test dataset
test_dataset = Dataset.from_pandas(test)
tokenized_test_dataset = test_dataset.map(preprocess_function, batched=True)


In [ ]:
# Check trainable parameters

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Define the compute metrics function
def compute_metrics(p):
    preds = torch.sigmoid(torch.tensor(p.predictions)).numpy()
    preds = (preds > 0.5).astype(int)
    labels = p.label_ids

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(labels, preds, average='micro')
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds, average='macro')
    accuracy = accuracy_score(labels, preds)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'precision_micro': precision_micro,
        'recall_micro': recall_micro,
        'f1_micro': f1_micro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
    }


In [ ]:
# Define training arguments
training_args = TrainingArguments(
    per_device_train_batch_size=10,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
# Predict on an example
example_text = test['text'][576]
encoded_example = tokenizer(
    example_text,
    truncation=True,
    padding="max_length",
    max_length=512,
    return_tensors="pt"
)

In [ ]:
# Evaluate the model
eval_results = trainer.evaluate()

# Print evaluation results
print("Evaluation results:")
for key, value in eval_results.items():
    print(f"{key}: {value}")
    
# Ensure the example is on the same device as the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoded_example = {key: value.to(device) for key, value in encoded_example.items()}

# Get model predictions
with torch.no_grad():
    outputs = model(**encoded_example)

# Apply sigmoid to get probabilities
logits = outputs['logits']
probs = torch.sigmoid(logits)

# Convert probabilities to binary predictions (0 or 1)
preds = (probs > 0.5).cpu().numpy().astype(int)

# Assuming you have a list of labels
label_list = [
    'Article 1: Obligation to respect human rights',
    'Article 2: Right to life',
    'Article 3: Prohibition of torture',
    'Article 4: Prohibition of slavery and forced labour',
    'Article 5: Right to liberty and security',
    'Article 6: Right to a fair trial',
    'Article 7: No punishment without law',
    'Article 8: Right to respect for private and family life',
    'Article 9: Freedom of thought, conscience and religion',
    'Article 10: Right to freedom of speech and expression',
    'No Articles Violated'
]

# Print predictions
for idx, label in enumerate(label_list):
    print(f"{label}: {'Yes' if preds[0][idx] else 'No'}")

In [ ]:
test['articles'][576]

In [ ]:
import os
import torch

# Define save directory
save_directory = "gemma2-2b finetuned"

# Create directory if it doesn't exist
if not os.path.exists(save_directory):
    os.makedirs(save_directory)

# Save the trained model's state dict
model_path = os.path.join(save_directory, "model.pth")
torch.save(model.state_dict(), model_path)

# Save the tokenizer
tokenizer.save_pretrained(save_directory)

print(f"Model and tokenizer saved to {save_directory}")
